# Laboratorio 4 – Árboles de Decisión

In [95]:
import pyreadr                          # Lectura de archivos .Rdata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento 
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Modelos 
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text, plot_tree
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans

# Métricas 
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Estilo gráfico
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

## 1. Carga de Datos

In [96]:
# 1.1 Lectura del archivo 
result = pyreadr.read_r('listings.Rdata')   # Cambiar ruta si es necesario
# Convierte el objeto R en un diccionario de DataFrames de Pandas sin necesidad de tener R instalado.
# El archivo puede contener uno o varios objetos R; tomamos el primero
df_raw = result[list(result.keys())[0]].copy()

print(f"Objeto(s) en el archivo: {list(result.keys())}")
print(f"\nDimensiones: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")


Objeto(s) en el archivo: ['listings']

Dimensiones: 171,748 filas x 80 columnas


In [97]:
# 1.2 Estructura del dataset
print("\nTIPOS DE DATOS ")
print(df_raw.dtypes.head(80))

print("\nPRIMERAS 5 FILAS")
df_raw.head()



TIPOS DE DATOS 
id                                              float64
listing_url                                      object
scrape_id                                       float64
last_scraped                                     object
source                                           object
                                                 ...   
calculated_host_listings_count_entire_homes       int32
calculated_host_listings_count_private_rooms      int32
calculated_host_listings_count_shared_rooms       int32
reviews_per_month                               float64
city                                             object
Length: 80, dtype: object

PRIMERAS 5 FILAS


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


In [98]:
# 1.3 Estadísticas descriptivas
df_raw.describe(include='number').T.style.background_gradient(cmap='Blues', axis=1)


,count,mean,std,min,25%,50%,75%,max
id,171748.000000,636291457352466304.000000,583492767102072064.000000,6.000000,35992997.750000,724955743120853504.000000,1182638277348256000.000000,1567804186654919936.000000
scrape_id,171748.000000,20251067780923.519531,141962486.493073,20250916040722.000000,20250916040734.000000,20251204025409.000000,20251204025459.000000,20251204025459.000000
host_id,171748.000000,200257103.404290,204882973.685601,23.000000,26330631.000000,108058504.500000,376574507.000000,732092326.000000
latitude,171748.000000,33.776320,7.236441,18.989648,30.381088,34.089840,40.702460,42.391844
longitude,171748.000000,-109.489530,30.024424,-159.716528,-118.591115,-117.865347,-73.997148,-70.996000
accommodates,171748.000000,4.106796,2.836661,1.000000,2.000000,4.000000,6.000000,16.000000
bathrooms,140352.000000,1.551795,1.014407,0.000000,1.000000,1.000000,2.000000,32.500000
minimum_nights,171748.000000,16.996559,30.014597,1.000000,2.000000,4.000000,30.000000,1125.000000
maximum_nights,171748.000000,12988.269540,5181839.672298,1.000000,90.000000,365.000000,1125.000000,2147483647.000000
minimum_nights_avg_ntm,171748.000000,17.487677,30.788964,0.700000,2.000000,4.400000,30.000000,1125.000000


### Interpretación de tipos de variables

| Tipo | Variables clave | Rol en el modelo |
|---|---|---|
| **Numéricas continuas** | `price`, `review_scores_rating`, `latitude`, `longitude` | Predictores y target |
| **Numéricas discretas** | `accommodates`, `bedrooms`, `beds`, `bathrooms` | Predictores clave |
| **Categóricas nominales** | `room_type`, `property_type`, `neighbourhood_cleansed` | Requieren encoding |
| **Booleanas** | `host_is_superhost`, `instant_bookable` | Binarias (0/1) |
| **Fechas** | `host_since`, `first_review`, `last_review` | Se extraerá antigüedad |
| **Texto libre / URLs** | `name`, `description`, `listing_url`, `picture_url` | Se eliminarán |
